In [5]:
import tempfile
import gradio as gr
import pandas as pd
from pyvis.network import Network
from rdflib import Graph, Literal, Namespace, RDF, RDFS, XSD

# --- 1. SETUP RDF & NAMESPACE ---
EX = Namespace("https://data.company-nis2.it/resource/")
ONT = Namespace("https://data.company-nis2.it/ontology/")


def create_rdf_graph(df: pd.DataFrame) -> Graph:
  g = Graph()
  g.bind("ex", EX)
  g.bind("ont", ONT)
  g.bind("rdfs", RDFS)

  for _, row in df.iterrows():
    piva = str(row["partita_iva"]).strip()
    c_uri = EX[f"company_{piva}"]
    ev_uri = EX[f"evidence_{piva}"]

    g.add((c_uri, RDF.type, ONT.Company))
    g.add((
        c_uri,
        RDFS.label,
        Literal(str(row["denominazione"]), datatype=XSD.string),
    ))
    g.add((c_uri, ONT.vatId, Literal(piva, datatype=XSD.string)))

    # NUOVO PREDICATO: Descrizione esplicita dell'attività reale svolta
    desc_reale = row.get("descrizione_attivita_reale", "")
    if desc_reale:
      g.add((
          c_uri,
          ONT.actualActivityDescription,
          Literal(str(desc_reale), datatype=XSD.string),
      ))

    # ATECO
    ateco = str(row.get("ateco_anagrafica_codice", "N/D")).strip()
    if ateco != "N/D":
      a_uri = EX[f"ateco_{ateco.replace('.', '_')}"]
      g.add((a_uri, RDF.type, ONT.AtecoCode))
      g.add((a_uri, ONT.code, Literal(ateco, datatype=XSD.string)))
      g.add((c_uri, ONT.hasOfficialAteco, a_uri))

      s_old = row.get("target_sector_originario")
      if s_old and str(s_old) not in ["Indefinito", "N/D"]:
        s_old_uri = EX[f"sector_{str(s_old).replace(' ', '_')}"]
        g.add((s_old_uri, RDF.type, ONT.BusinessSector))
        g.add(
            (s_old_uri, RDFS.label, Literal(str(s_old), datatype=XSD.string))
        )
        g.add((a_uri, ONT.historicallyMappedTo, s_old_uri))

    # Evidenza e Settore Finale
    g.add((ev_uri, RDF.type, ONT.ActivityEvidence))
    g.add((
        ev_uri,
        ONT.activitySummary,
        Literal(str(row.get("sintesi_attivita_reale", ""))),
    ))
    g.add((c_uri, ONT.conductsActivity, ev_uri))

    s_new = row.get("settore_suggerito")
    if s_new:
      s_new_uri = EX[f"sector_{str(s_new).replace(' ', '_')}"]
      g.add((s_new_uri, RDF.type, ONT.BusinessSector))
      g.add((s_new_uri, RDFS.label, Literal(str(s_new))))
      g.add((ev_uri, ONT.suggestsSector, s_new_uri))

    # Settore NIS Primario
    nis_p = row.get("settore_nis_primario")
    if nis_p and nis_p != "Fuori Perimetro NIS 2":
      nis_uri = EX[f"nis_{str(nis_p).replace(' ', '_')[:30]}"]
      g.add((nis_uri, RDF.type, ONT.NISSector))
      g.add((nis_uri, RDFS.label, Literal(str(nis_p))))
      g.add((c_uri, ONT.hasPrimaryNISSector, nis_uri))

  return g


# --- 2. DATI INITIAL_DATA AGGIORNATI ---
initial_data = [
    {
        "denominazione": "FUMAKILLA EUROPE S.R.L.",
        "partita_iva": "03456780961",
        "ateco_anagrafica_codice": "46.46.3",
        "target_sector_originario": "Commercio",
        "descrizione_attivita_reale": "L'azienda produce insetticidi e formulati chimici per la disinfestazione domestica e industriale.",
        "sintesi_attivita_reale": (
            "Produzione chimica industriale di insetticidi e repellenti."
        ),
        "settore_suggerito": "Manifatturiero",
        "settore_nis_primario": (
            "Produzione e distribuzione di sostanze chimiche"
        ),
        "discrepanza_rilevata": True,
    },
    {
        "denominazione": "CLOUD & DEFENSE SOLUTIONS S.P.A.",
        "partita_iva": "08912340582",
        "ateco_anagrafica_codice": "62.01.0",
        "target_sector_originario": "Tecnologico",
        "descrizione_attivita_reale": "L'azienda sviluppa software di difesa cibernetica e gestisce data center per infrastrutture critiche.",
        "sintesi_attivita_reale": (
            "Erogazione di servizi cloud e data center per infrastrutture critiche."
        ),
        "settore_suggerito": "Tecnologico",
        "settore_nis_primario": "Fornitori di servizi cloud computing",
        "discrepanza_rilevata": False,
    },
    {
        "denominazione": "ARES BOXING CLUB ASSOCIAZIONE SPORTIVA DILETTANTISTICA",
        "partita_iva": "97654320153",
        "ateco_anagrafica_codice": "93.12.0",
        "target_sector_originario": "Intrattenimento",
        "descrizione_attivita_reale": "L'ente gestisce corsi di pugilato e discipline sportive a livello amatoriale e agonistico senza fini di lucro.",
        "sintesi_attivita_reale": "Associazione sportiva dilettantistica no-profit.",
        "settore_suggerito": "Organizzazioni associative",
        "settore_nis_primario": "Fuori Perimetro NIS 2",
        "discrepanza_rilevata": True,
    },
]

current_df = pd.DataFrame(initial_data)
current_graph = create_rdf_graph(current_df)

In [6]:
# --- 3. HELPER PER VISUALIZZAZIONE GRAFO CON PYVIS ---
import html
import tempfile
from pyvis.network import Network


def render_pyvis_network(piva_filter: str = None) -> str:
  net = Network(height="550px", width="100%", directed=True, bgcolor="#ffffff")

  df_view = (
      current_df[current_df["partita_iva"] == piva_filter]
      if piva_filter and piva_filter != "Tutte"
      else current_df
  )

  for _, r in df_view.iterrows():
    c_id = f"c_{r['partita_iva']}"
    net.add_node(
        c_id,
        label=r["denominazione"][:22] + "...",
        title=f"{r['denominazione']} (P.IVA: {r['partita_iva']})",
        color="#2b5c8f",
        size=22,
    )

    # ATECO
    ateco = str(r["ateco_anagrafica_codice"])
    net.add_node(ateco, label=f"ATECO {ateco}", color="#6c757d", shape="box")
    net.add_edge(c_id, ateco, label="hasOfficialAteco")

    # Settore Originario DB
    s_old = str(r["target_sector_originario"])
    net.add_node(s_old, label=f"DB: {s_old}", color="#dc3545", shape="ellipse")
    net.add_edge(c_id, s_old, label="historicallyTaggedAs")

    # Settore Assegnato
    s_new = str(r["settore_suggerito"])
    net.add_node(
        s_new, label=f"Finale: {s_new}", color="#198754", shape="ellipse"
    )

    ev_id = f"ev_{r['partita_iva']}"
    net.add_node(
        ev_id,
        label="Evidenza Attività",
        title=r["sintesi_attivita_reale"],
        color="#ffc107",
        shape="dot",
    )
    net.add_edge(c_id, ev_id, label="conductsActivity")
    net.add_edge(ev_id, s_new, label="suggestsSector")

    # Settore NIS
    nis = str(r["settore_nis_primario"])
    if nis != "Fuori Perimetro NIS 2":
      net.add_node(nis, label=f"NIS: {nis}", color="#0dcaf0", shape="box")
      net.add_edge(c_id, nis, label="hasPrimaryNISSector")

  net.set_options(
      '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -50}},'
      ' "edges": {"smooth": false}}'
  )

  # Salviamo il file HTML generato da pyvis
  tmp_file = tempfile.NamedTemporaryFile(
      delete=False, suffix=".html", mode="w", encoding="utf-8"
  )
  net.save_graph(tmp_file.name)

  # Leggiamo il contenuto HTML
  with open(tmp_file.name, "r", encoding="utf-8") as f:
    html_content = f.read()

  # TRUCCO: Inseriamo il codice dentro un <iframe> con srcdoc escaped,
  # consentendo l'esecuzione di vis.js senza essere bloccato da Gradio!
  escaped_content = html.escape(html_content)
  iframe_html = f"""
    <iframe
        style="width: 100%; height: 600px; border: 1px solid #e0e0e0; border-radius: 8px;"
        srcdoc="{escaped_content}"
        frameborder="0">
    </iframe>
    """
  return iframe_html

def get_company_subgraph_sparql(vat_id: str) -> dict:
  global current_graph

  vat_id_clean = str(vat_id).strip()
  if not vat_id_clean or vat_id_clean == "Nessuna selezione":
    return None

  query = f"""
    PREFIX ex: <https://data.company-nis2.it/resource/>
    PREFIX ont: <https://data.company-nis2.it/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?company ?denominazione ?actualDesc ?ateco ?oldSector ?newSector ?evidence ?nisSector
    WHERE {{
        ?company a ont:Company ;
                 rdfs:label ?denominazione ;
                 ont:vatId ?vatId .
        
        FILTER(str(?vatId) = "{vat_id_clean}")

        OPTIONAL {{ ?company ont:actualActivityDescription ?actualDesc . }}

        # 1. Recupero ATECO e il Settore Storico DB
        OPTIONAL {{
            ?company ont:hasOfficialAteco ?a .
            ?a ont:code ?ateco .
            OPTIONAL {{
                ?a ont:historicallyMappedTo ?sOld .
                ?sOld rdfs:label ?oldSector .
            }}
        }}

        # 2. Recupero Evidenza e Settore Finale Suggerito
        OPTIONAL {{
            ?company ont:conductsActivity ?ev .
            OPTIONAL {{ ?ev ont:activitySummary ?evidence . }}
            OPTIONAL {{
                ?ev ont:suggestsSector ?sNew .
                ?sNew rdfs:label ?newSector .
            }}
        }}

        # 3. Settore NIS 2
        OPTIONAL {{
            ?company ont:hasPrimaryNISSector ?nis .
            ?nis rdfs:label ?nisSector .
        }}
    }}
    """
  results = list(current_graph.query(query))
  if not results:
    return None

  row = results[0]
  return {
      "company_uri": str(row.company),
      "denominazione": str(row.denominazione) if row.denominazione is not None else "N/D",
      "vat_id": vat_id_clean,
      "actual_desc": str(row.actualDesc) if row.actualDesc is not None else "Non dichiarata",
      "ateco": str(row.ateco) if row.ateco is not None else "N/D",
      "old_sector": str(row.oldSector) if row.oldSector is not None else "N/D",
      "new_sector": str(row.newSector) if row.newSector is not None else "N/D",
      "evidence": str(row.evidence) if row.evidence is not None else "Nessuna evidenza registrata",
      "nis_sector": str(row.nisSector) if row.nisSector is not None else "N/D",
  }


def render_sparql_single_entity_graph(vat_id: str) -> str:
  if not vat_id or vat_id == "Nessuna selezione":
    return "<div style='padding:20px; text-align:center;'>Seleziona una Partita IVA per visualizzare il grafo.</div>"

  data = get_company_subgraph_sparql(vat_id)
  if not data:
    return f"<div style='padding:20px;'>Nessun dato trovato nel grafo RDF per P.IVA: {vat_id}</div>"

  net = Network(height="500px", width="100%", directed=True, bgcolor="#ffffff")

  # Nodo Centrale: Company (il tooltip ora include anche l'attività reale completa)
  c_node_id = f"comp_{vat_id}"
  net.add_node(
      c_node_id,
      label=data["denominazione"][:25] + "...",
      title=f"{data['denominazione']}\nP.IVA: {vat_id}\nAttività Reale: {data['actual_desc']}",
      color="#2b5c8f",
      size=28,
      shape="ellipse",
  )

  # Nodo ATECO Ufficiale & Settore Storico DB
  ateco = data["ateco"]
  s_old = data["old_sector"]
  if ateco and ateco != "N/D":
    net.add_node(ateco, label=f"ATECO: {ateco}", color="#6c757d", shape="box")
    net.add_edge(c_node_id, ateco, label="ont:hasOfficialAteco")

    if s_old and s_old != "N/D":
      net.add_node(s_old, label=f"DB: {s_old}", color="#dc3545", shape="ellipse")
      net.add_edge(ateco, s_old, label="ont:historicallyMappedTo")

  # Nodo Evidenza Attività + Settore Suggerito
  if data["evidence"]:
    ev_id = f"ev_{vat_id}"
    net.add_node(
        ev_id,
        label="Evidenza Attività",
        title=f"Sintesi: {data['evidence']}",
        color="#ffc107",
        shape="dot",
        size=18,
    )
    net.add_edge(c_node_id, ev_id, label="ont:conductsActivity")

    if data["new_sector"]:
      new_id = f"new_{data['new_sector']}"
      net.add_node(
          new_id,
          label=f"Settore Finale: {data['new_sector']}",
          color="#198754",
          shape="ellipse",
      )
      net.add_edge(ev_id, new_id, label="ont:suggestsSector")

  # Settore NIS 2
  if data["nis_sector"]:
    nis_id = f"nis_{data['nis_sector']}"
    net.add_node(
        nis_id,
        label=f"NIS 2: {data['nis_sector']}",
        color="#0dcaf0",
        shape="box",
    )
    net.add_edge(c_node_id, nis_id, label="ont:hasPrimaryNISSector")

  net.set_options(
      '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -70}}, "edges": {"smooth": false}}'
  )

  tmp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".html", mode="w", encoding="utf-8")
  net.save_graph(tmp_file.name)
  with open(tmp_file.name, "r", encoding="utf-8") as f:
    html_content = f.read()

  escaped_content = html.escape(html_content)
  return f"""
    <iframe
        style="width: 100%; height: 550px; border: 1px solid #e0e0e0; border-radius: 8px;"
        srcdoc="{escaped_content}"
        frameborder="0">
    </iframe>
    """


def get_entity_details_sparql(vat_id: str):
  data = get_company_subgraph_sparql(vat_id)
  if not data:
    return "", "", "", "", "", ""
  return (
      data["denominazione"],
      data["actual_desc"],
      data["ateco"] or "N/D",
      data["old_sector"] or "N/D",
      data["new_sector"] or "N/D",
      data["evidence"] or "Nessuna evidenza registrata",
  )


def render_sparql_single_entity_graph(vat_id: str) -> str:
  """Costruisce la visualizzazione PyVis basandosi ESCLUSIVAMENTE

  sulle relazioni estratte via SPARQL dal grafo RDF.
  """
  if not vat_id or vat_id == "Nessuna selezione":
    return "<div style='padding:20px; text-align:center;'>Seleziona una Partita IVA per visualizzare il grafo.</div>"

  data = get_company_subgraph_sparql(vat_id)
  if not data:
    return f"<div style='padding:20px;'>Nessun dato trovato nel grafo RDF per P.IVA: {vat_id}</div>"

  net = Network(height="500px", width="100%", directed=True, bgcolor="#ffffff")

  # Nodo Centrale: Company
  c_node_id = f"comp_{vat_id}"
  net.add_node(
      c_node_id,
      label=data["denominazione"][:25] + "...",
      title=f"{data['denominazione']}\nP.IVA: {vat_id}",
      color="#2b5c8f",
      size=28,
      shape="ellipse",
  )

  # Nodo ATECO Ufficiale
# CODICE AGGIORNATO:
    # 1. Nodo ATECO
  ateco = data["ateco"]
  s_old = data["old_sector"]  
  c_id = f"comp_{data['vat_id']}"
  
  if ateco and ateco != "N/D":
      net.add_node(ateco, label=f"ATECO: {ateco}", color="#6c757d", shape="box")
      net.add_edge(c_id, ateco, label="ont:hasOfficialAteco")

      # 2. Nodo Settore Storico DB (COLLEGATO ALL'ATECO, NON ALL'AZIENDA)
      if s_old and s_old != "N/D":
        net.add_node(
            s_old, label=f"DB: {s_old}", color="#dc3545", shape="ellipse"
        )
        net.add_edge(ateco, s_old, label="ont:historicallyMappedTo")

  # Nodo Evidenza Attività + Settore Suggerito
  if data["evidence"]:
    ev_id = f"ev_{vat_id}"
    net.add_node(
        ev_id,
        label="Evidenza Attività",
        title=data["evidence"],
        color="#ffc107",
        shape="dot",
        size=18,
    )
    net.add_edge(c_node_id, ev_id, label="ont:conductsActivity")

    if data["new_sector"]:
      new_id = f"new_{data['new_sector']}"
      net.add_node(
          new_id,
          label=f"Settore Finale: {data['new_sector']}",
          color="#198754",
          shape="ellipse",
      )
      net.add_edge(ev_id, new_id, label="ont:suggestsSector")

  # Nodo Settore NIS 2
  if data["nis_sector"]:
    nis_id = f"nis_{data['nis_sector']}"
    net.add_node(
        nis_id,
        label=f"NIS 2: {data['nis_sector']}",
        color="#0dcaf0",
        shape="box",
    )
    net.add_edge(c_node_id, nis_id, label="ont:hasPrimaryNISSector")

  net.set_options(
      '{"physics": {"forceAtlas2Based": {"gravitationalConstant": -70}},'
      ' "edges": {"smooth": false}}'
  )

  tmp_file = tempfile.NamedTemporaryFile(
      delete=False, suffix=".html", mode="w", encoding="utf-8"
  )
  net.save_graph(tmp_file.name)
  with open(tmp_file.name, "r", encoding="utf-8") as f:
    html_content = f.read()

  escaped_content = html.escape(html_content)
  return f"""
    <iframe
        style="width: 100%; height: 550px; border: 1px solid #e0e0e0; border-radius: 8px;"
        srcdoc="{escaped_content}"
        frameborder="0">
    </iframe>
    """


def get_entity_details_sparql(vat_id: str):
  """Estrae i campi descrittivi formattati per i box di testo della UI."""
  data = get_company_subgraph_sparql(vat_id)
  if not data:
    return "", "", "", "", ""
  return (
      data["denominazione"],
      data["ateco"] or "N/D",
      data["old_sector"] or "N/D",
      data["new_sector"] or "N/D",
      data["evidence"] or "Nessuna evidenza registrata",
  )

# --- 4. FUNZIONE DI INFERENZA SINGOLA (MOCK AGENTE) ---
def classify_single(nome, piva, ateco, target_db):
  global current_df, current_graph
  # Logica mock coerente con le regole del sistema
  if any(w in nome.lower() for w in ["boxing", "sport", "asd", "aps"]):
    settore_sugg = "Organizzazioni associative"
    nis_p = "Fuori Perimetro NIS 2"
    sintesi = "Ente associativo sportivo dilettantistico senza scopo di lucro."
  elif (
      "chimica" in nome.lower()
      or "fumakilla" in nome.lower()
      or ateco.startswith("20")
  ):
    settore_sugg = "Manifatturiero"
    nis_p = "Produzione e distribuzione di sostanze chimiche"
    sintesi = (
        "Produzione industriale e sintesi chimica; prevale su ATECO commerciale."
    )
  else:
    settore_sugg = "Tecnologico"
    nis_p = "Fornitori di servizi digitali"
    sintesi = "Attività IT/tecnologica accertata tramite canali digitali."

  discrepanza = str(target_db) != settore_sugg

  new_row = {
      "denominazione": nome,
      "partita_iva": piva,
      "ateco_anagrafica_codice": ateco,
      "target_sector_originario": target_db,
      "sintesi_attivita_reale": sintesi,
      "settore_suggerito": settore_sugg,
      "settore_nis_primario": nis_p,
      "discrepanza_rilevata": discrepanza,
  }

  current_df = pd.concat([current_df, pd.DataFrame([new_row])],
                         ignore_index=True)
  current_graph = create_rdf_graph(current_df)

  return (
      settore_sugg,
      nis_p,
      sintesi,
      "DISCREPANZA RILEVATA (Divergenza dal DB)"
      if discrepanza
      else "COERENTE CON IL DB",
      current_df,
      gr.update(choices=["Tutte"] + current_df["partita_iva"].tolist()),
  )




In [7]:
fumi_vat="03456780961"
x=get_company_subgraph_sparql(fumi_vat)
x

{'company_uri': 'https://data.company-nis2.it/resource/company_03456780961',
 'denominazione': 'FUMAKILLA EUROPE S.R.L.',
 'vat_id': '03456780961',
 'actual_desc': "L'azienda produce insetticidi e formulati chimici per la disinfestazione domestica e industriale.",
 'ateco': '46.46.3',
 'old_sector': 'Commercio',
 'new_sector': 'Manifatturiero',
 'evidence': 'Produzione chimica industriale di insetticidi e repellenti.',
 'nis_sector': 'Produzione e distribuzione di sostanze chimiche'}

In [8]:
# --- 5. INTERFACCIA GRADIO ---
with gr.Blocks(title="NIS 2 & Business Sector Classification KG") as demo:
  gr.Markdown("# 🛡️ NIS 2 & Business Sector Classification Platform")
  gr.Markdown(
      "Control Room Multi-Agente con generazione integrata di **Knowledge"
      " Graph (RDF)**."
  )

  with gr.Tab("📋 Batch Table & Status"):
    table_view = gr.DataFrame(
        value=current_df,
        headers=list(current_df.columns),
        interactive=False,
        wrap=True,
    )
    with gr.Row():
      refresh_btn = gr.Button("🔄 Aggiorna Tabella")
      download_ttl_btn = gr.Button("💾 Esporta Grafo Turtle (.ttl)")
    ttl_download_file = gr.File(label="Scarica File RDF")

    def export_ttl():
      current_graph.serialize(
          destination="export_graph.ttl", format="turtle"
      )
      return "export_graph.ttl"

    download_ttl_btn.click(export_ttl, outputs=ttl_download_file)
    refresh_btn.click(lambda: current_df, outputs=table_view)

  with gr.Tab("🔍 Single Entity Inspector"):
    with gr.Row():
      with gr.Column():
        in_nome = gr.Textbox(
            label="Denominazione", value="SOLUZIONI CHIMICHE INDUSTRIALI S.R.L."
        )
        in_piva = gr.Textbox(label="Partita IVA", value="09988776655")
        in_ateco = gr.Textbox(label="Codice ATECO", value="46.75.0")
        in_db = gr.Textbox(
            label="Settore Storico a DB (target_sector)", value="Commercio"
        )
        submit_btn = gr.Button("🚀 Esegui Valutazione Pipeline", variant="primary")
      with gr.Column():
        out_settore = gr.Textbox(label="Settore Assegnato (22 ammessi)")
        out_nis = gr.Textbox(label="Inquadramento Settore NIS 2")
        out_status = gr.Textbox(label="Confronto con Storico DB")
        out_sintesi = gr.TextArea(label="Sintesi Attività Reale (Audit Trail)")

  with gr.Tab("🕸️ Knowledge Graph Inspector"):
    with gr.Row():
      piva_dropdown = gr.Dropdown(
          choices=["Tutte"] + current_df["partita_iva"].tolist(),
          value="Tutte",
          label="Filtra per Soggetto (Partita IVA)",
      )
      render_btn = gr.Button("Aggiorna Grafo", variant="secondary")

    html_graph = gr.HTML(value=render_pyvis_network())

    piva_dropdown.change(render_pyvis_network, inputs=piva_dropdown, outputs=html_graph)
    render_btn.click(render_pyvis_network, inputs=piva_dropdown, outputs=html_graph)

  with gr.Tab("🎯 SPARQL Entity Inspector"):
    gr.Markdown(
        "### Visualizzazione Puntuale Soggetto (Risolto tramite Query SPARQL"
        " su RDF)"
    )

    with gr.Row():
      sparql_piva_selector = gr.Dropdown(
          choices=current_df["partita_iva"].tolist(),
          value=current_df["partita_iva"].tolist()[0]
          if not current_df.empty
          else None,
          label="Seleziona Soggetto (Partita IVA)",
          interactive=True,
      )
      load_sparql_btn = gr.Button("🔍 Esegui SPARQL & Mostra Grafo")

    with gr.Row():
      with gr.Column(scale=1):
        sp_name = gr.Textbox(label="Denominazione (rdfs:label)", interactive=False)
        sp_ateco = gr.Textbox(label="Codice ATECO (ont:hasOfficialAteco)", interactive=False)
        sp_old_sec = gr.Textbox(label="Settore DB Storico (ont:historicallyTaggedAs)", interactive=False)
        sp_new_sec = gr.Textbox(label="Settore Assegnato (ont:suggestsSector)", interactive=False)
        sp_evidence = gr.TextArea(label="Evidenza Attività (ont:activitySummary)", lines=4, interactive=False)

      with gr.Column(scale=2):
        sp_graph_html = gr.HTML(
            value=render_sparql_single_entity_graph(
                current_df["partita_iva"].tolist()[0]
                if not current_df.empty
                else ""
            )
        )

      with gr.Row():
        with gr.Column(scale=1):
          sp_name = gr.Textbox(label="Denominazione (rdfs:label)", interactive=False)
          sp_desc = gr.TextArea(label="Attività Reale (ont:actualActivityDescription)", lines=2, interactive=False)
          sp_ateco = gr.Textbox(label="Codice ATECO (ont:hasOfficialAteco)", interactive=False)
          sp_old_sec = gr.Textbox(label="Settore DB Storico (ont:historicallyMappedTo)", interactive=False)
          sp_new_sec = gr.Textbox(label="Settore Assegnato (ont:suggestsSector)", interactive=False)
          sp_evidence = gr.TextArea(label="Evidenza Attività (ont:activitySummary)", lines=3, interactive=False)

        with gr.Column(scale=2):
          sp_graph_html = gr.HTML(
              value=render_sparql_single_entity_graph(
                  current_df["partita_iva"].tolist()[0] if not current_df.empty else ""
              )
          )

    def on_sparql_entity_selected(vat_id):
      name, desc, ateco, old_s, new_s, evid = get_entity_details_sparql(vat_id)
      graph_view = render_sparql_single_entity_graph(vat_id)
      return name, desc, ateco, old_s, new_s, evid, graph_view

    sparql_piva_selector.change(
        on_sparql_entity_selected,
        inputs=[sparql_piva_selector],
        outputs=[sp_name, sp_desc, sp_ateco, sp_old_sec, sp_new_sec, sp_evidence, sp_graph_html],
    )
    load_sparql_btn.click(
        on_sparql_entity_selected,
        inputs=[sparql_piva_selector],
        outputs=[sp_name, sp_desc, sp_ateco, sp_old_sec, sp_new_sec, sp_evidence, sp_graph_html],
    )

    # Eventi al cambio della dropdown o click del pulsante
    def on_sparql_entity_selected(vat_id):
      name, ateco, old_s, new_s, evid = get_entity_details_sparql(vat_id)
      graph_view = render_sparql_single_entity_graph(vat_id)
      return name, ateco, old_s, new_s, evid, graph_view

    sparql_piva_selector.change(
        on_sparql_entity_selected,
        inputs=[sparql_piva_selector],
        outputs=[sp_name, sp_ateco, sp_old_sec, sp_new_sec, sp_evidence, sp_graph_html],
    )
    load_sparql_btn.click(
        on_sparql_entity_selected,
        inputs=[sparql_piva_selector],
        outputs=[sp_name, sp_ateco, sp_old_sec, sp_new_sec, sp_evidence, sp_graph_html],
    )  

  submit_btn.click(
      classify_single,
      inputs=[in_nome, in_piva, in_ateco, in_db],
      outputs=[
          out_settore,
          out_nis,
          out_sintesi,
          out_status,
          table_view,
          piva_dropdown,
      ],
  )

if __name__ == "__main__":
  demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [ ]:

            st.rerun()

        if modifica_btn:
            st.session_state.records[selected_idx]["settore_suggerito"] = nuovo_settore
            st.session_state.records[selected_idx]["settore_nis_primario"] = nuovo_nis
            st.session_state.records[selected_idx]["stato_revisione"] = "Modificato"
            st.session_state.records[selected_idx]["note_operatore"] = note or "Modificato manualmente dall'operatore"
            st.success("Modifiche salvate e grafo aggiornato!")
            st.rerun()

with col_right:
    badge_color = {
        "In attesa": "orange",
        "Confermato": "green",
        "Modificato": "blue",
    }.get(curr_rec["stato_revisione"], "gray")
    
    st.subheader("Subgrafo Semantico RDF (PyVis)")
    st.markdown(f"**Stato Validazione Attuale:** :{badge_color}[{curr_rec['stato_revisione']}]")

    # Render del subgrafo
    graph_html = render_pyvis(curr_rec)
    components.html(graph_html, height=440)

# --- TABELLA RIASSUNTIVA E DOWNLOAD TURTLE (.TTL) ---
st.divider()
st.subheader("Riepilogo Revisioni & Export")

df_current = pd.DataFrame(st.session_state.records)
st.dataframe(
    df_current[[
        "denominazione", "partita_iva", "descrizione_attivita_reale", 
        "settore_suggerito", "settore_nis_primario", "stato_revisione", "note_operatore"
    ]],
    use_container_width=True
)

# Sincronizzazione ed export del grafo Turtle
current_rdf_graph = create_rdf_graph(df_current)
ttl_data = current_rdf_graph.serialize(format="turtle")

st.download_button(
    label="💾 Esporta Grafo Aggiornato (.ttl)",
    data=ttl_data,
    file_name="kg_validated_data.ttl",
    mime="text/turtle"
)

2026-11-06 08:33:43.834 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-06 08:33:43.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

False